## AgroManager

Build AgroManagement file for simulations of selected marginal NUTS3 regions across Europe. Simulations are intended for current marginal land conditions across Europe, as defined in MIDAS (2019-2023)

In [4]:
import numpy as np
import pandas as pd
from pathlib import Path
import yaml
import os 

In [5]:
# Create output directory
folder = Path("/Users/paulianoprescu/Projects/wofost_miscanthus/marginal_simulations_eu")
output_dir = folder / "agromanager" 

In [6]:
# Define Agromanagement Assumptions
# Hereby, this notebook is reproduceable, assumptions are well documented, and if there is a change needed, everything can update accordingly
#
# Since Miscanthus is a perennial, each growing season is modelled as a different campaign 

crop_name = "miscanthus"
variety_name = "miscanthus_sinensis"
crop_start_type = "emergence"   # rhizome sprouting = emergence; have not chosen for sowing
crop_end_type = "harvest"  

# Growing Season Window (month, day)
start_month, start_day = 1, 1   # My observation in 2026 was that harvest happened in late March / early April; hereby I assume this emergance date 
end_month, end_day = 12, 30    # At this point in time I assume the crop to be fully grown

# Years to simulate [CURRENT MIDAS MARGINAL LAND CONDITIONS]
years = list(range(2019, 2024))

# Year 1: campaign starts 90 days early for spin-up
first_year = years[0]
first_campaign = f"""- {first_year - 1}-10-01:
    CropCalendar:
        crop_name: {crop_name}
        variety_name: {variety_name}
        crop_start_date: {first_year}-{start_month:02d}-{start_day:02d}
        crop_start_type: {crop_start_type}
        crop_end_date: {first_year}-{end_month:02d}-{end_day:02d}
        crop_end_type: {crop_end_type}
        max_duration: null
    TimedEvents: null
    StateEvents: null"""

# Subsequent years: campaign starts on crop_start_date (no extra spin-up)
later_campaigns = "\n".join([f"""- {year}-{start_month:02d}-{start_day:02d}:
    CropCalendar:
        crop_name: {crop_name}
        variety_name: {variety_name}
        crop_start_date: {year}-{start_month:02d}-{start_day:02d}
        crop_start_type: {crop_start_type}
        crop_end_date: {year}-{end_month:02d}-{end_day:02d}
        crop_end_type: {crop_end_type}
        max_duration: null
    TimedEvents: null
    StateEvents: null""" for year in years[1:]])

campaigns = first_campaign + "\n" + later_campaigns

miscanthus_agro = f"""Version: 1.0.0
AgroManagement:
{campaigns}
"""

os.makedirs(output_dir, exist_ok=True)
outpath = output_dir / f"misc_agro.yaml"
with open(outpath, 'w') as f:
    f.write(miscanthus_agro)
print(f"Miscanthus AgroManagement CropCalender file written to: {outpath}")

Miscanthus AgroManagement CropCalender file written to: /Users/paulianoprescu/Projects/wofost_miscanthus/marginal_simulations_eu/agromanager/misc_agro.yaml
